<a href="https://colab.research.google.com/github/leovarconrenno/Estacao-de-Reabastecimento-de-Hidrogenio---SCADA-Core/blob/main/etapa-01-logica/10%20-%20Avaliacao%20Modulo%201%20Motor%20de%20Intertravamento%20e%20Diagnostico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 10 - Notebook: Avaliação Integrada do Módulo 1 — SCADA-Core Segurança & Diagnóstico

Este notebook integra todos os componentes desenvolvidos no **Módulo 1** para a Estação de Reabastecimento de Hidrogênio: telemetria de sensores (conversão de leituras físicas em proposições lógicas), Base de Conhecimento (Aula 08) e Motor de Inferência *Forward Chaining* (Aula 09), consolidados num único ciclo de varredura (*scan cycle*) do SCADA-Core.

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Any


class MapeadorProposicional:
    """Converte a telemetria bruta (valores físicos lidos dos instrumentos)
    em proposições lógicas binárias, usando os limiares definidos na Aula 02
    (Mapeamento de Variáveis de Processo)."""

    def extrair_proposicoes(self, telemetria: Dict[str, float]) -> Dict[str, bool]:
        return {
            # --- Setor 100: Armazenamento ---
            'p1_1': telemetria.get('PT-101', 0.0) >= 400.0,     # Tanque baixa pressão
            'v1_1': bool(telemetria.get('XV-101', 0)),
            'g1_1': telemetria.get('AT-101', 0.0) >= 25.0,

            'p1_2': telemetria.get('PT-102', 0.0) >= 700.0,     # Tanque média pressão
            'v1_2': bool(telemetria.get('XV-102', 0)),
            'g1_2': telemetria.get('AT-102', 0.0) >= 25.0,

            'p1_3': telemetria.get('PT-103', 0.0) >= 1000.0,    # Tanque alta pressão
            'v1_3': bool(telemetria.get('XV-103', 0)),
            'g1_3': telemetria.get('AT-103', 0.0) >= 25.0,

            'e1_1': bool(telemetria.get('ESD-100', 0)),

            # --- Setor 200: Condicionamento ---
            'nc_201': telemetria.get('TT-201', -273.0) > -40.0,  # chiller fora da faixa
            'h3_1': bool(telemetria.get('HS-301', 0)),

            # --- Setor 300: Dispensação ---
            'g3_1': telemetria.get('AT-301', 0.0) >= 25.0,
            't3_1': telemetria.get('TT-301', 0.0) >= 85.0,
            'p3_1': telemetria.get('PT-301', 0.0) >= 700.0,
        }


@dataclass
class RegraProducao:
    id_regra: str
    antecedentes: Set[str]
    consequente: str
    descricao_diagnostico: str
    prioridade: int = 1


class BaseConhecimento:
    def __init__(self):
        self.regras: List[RegraProducao] = []

    def adicionar_regra(self, id_r, antecedentes, consequente, desc, prioridade=1):
        self.regras.append(RegraProducao(id_r, set(antecedentes), consequente, desc, prioridade))


class MotorInferencia:
    def __init__(self, base_conhecimento):
        self.bc = base_conhecimento

    def forward_chaining(self, fatos_iniciais):
        fatos_conhecidos = set(fatos_iniciais)
        historico = []
        passo = 1
        novos = True
        while novos:
            novos = False
            for regra in sorted(self.bc.regras, key=lambda r: r.prioridade, reverse=True):
                if regra.antecedentes.issubset(fatos_conhecidos) and regra.consequente not in fatos_conhecidos:
                    fatos_conhecidos.add(regra.consequente)
                    historico.append({"Passo": passo, "Regra": regra.id_regra, "Diagnóstico": regra.descricao_diagnostico})
                    passo += 1
                    novos = True
                    break
        return fatos_conhecidos, historico


class SCADACoreModulo1:
    """Integra Mapeador + Base de Conhecimento + Motor de Inferência
    num único ciclo de varredura (scan) da Estação de Reabastecimento de H₂."""

    def __init__(self):
        self.mapeador = MapeadorProposicional()
        self.bc = BaseConhecimento()

        # --- Setor 100: Armazenamento ---
        self.bc.adicionar_regra("R-01", ["p1_1"], "SOBREPRESSAO_TANQUE_BAIXA", "Pressão do Tanque de Baixa Pressão acima do limite (PT-101 > 400 bar)", 9)
        self.bc.adicionar_regra("R-02", ["SOBREPRESSAO_TANQUE_BAIXA", "v1_1"], "TRIP_TANQUE_BAIXA", "Corte de Segurança do Tanque de Baixa Pressão", 10)
        self.bc.adicionar_regra("R-03", ["g1_1"], "FUGA_H2_TANQUE_BAIXA", "Vazamento de H₂ na área do Tanque de Baixa Pressão (AT-101)", 10)

        self.bc.adicionar_regra("R-04", ["p1_2"], "SOBREPRESSAO_TANQUE_MEDIA", "Pressão do Tanque de Média Pressão acima do limite (PT-102 > 700 bar)", 9)
        self.bc.adicionar_regra("R-05", ["SOBREPRESSAO_TANQUE_MEDIA", "v1_2"], "TRIP_TANQUE_MEDIA", "Corte de Segurança do Tanque de Média Pressão", 10)
        self.bc.adicionar_regra("R-06", ["g1_2"], "FUGA_H2_TANQUE_MEDIA", "Vazamento de H₂ na área do Tanque de Média Pressão (AT-102)", 10)

        self.bc.adicionar_regra("R-07", ["p1_3"], "SOBREPRESSAO_TANQUE_ALTA", "Pressão do Tanque de Alta Pressão acima do limite (PT-103 > 1000 bar)", 9)
        self.bc.adicionar_regra("R-08", ["SOBREPRESSAO_TANQUE_ALTA", "v1_3"], "TRIP_TANQUE_ALTA", "Corte de Segurança do Tanque de Alta Pressão", 10)
        self.bc.adicionar_regra("R-09", ["g1_3"], "FUGA_H2_TANQUE_ALTA", "Vazamento de H₂ na área do Tanque de Alta Pressão (AT-103)", 10)

        self.bc.adicionar_regra("R-10", ["e1_1"], "PARADA_EMERGENCIA_GERAL", "Parada de Emergência acionada manualmente pelo operador (ESD-100)", 10)

        # --- Setor 200: Condicionamento ---
        self.bc.adicionar_regra("R-11", ["nc_201", "h3_1"], "BLOQUEIO_DISPENSACAO_TEMPERATURA", "Início de abastecimento solicitado com pré-resfriamento fora da faixa (TT-201 > -40°C)", 6)

        # --- Setor 300: Dispensação ---
        self.bc.adicionar_regra("R-12", ["g3_1"], "FUGA_H2_DISPENSADOR", "Vazamento de H₂ detectado na área do dispensador (AT-301)", 10)
        self.bc.adicionar_regra("R-13", ["t3_1"], "SOBRETEMPERATURA_RECEPCAO_VEICULO", "Temperatura no ponto de recepção do veículo acima do limite (TT-301 > 85°C)", 8)
        self.bc.adicionar_regra("R-14", ["p3_1"], "ABASTECIMENTO_CONCLUIDO", "Pressão de enchimento atinge o setpoint do veículo (PT-301 ≈ 700 bar)", 2)

        self.motor = MotorInferencia(self.bc)

        # Fatos primitivos que, por si só, já caracterizam uma condição crítica
        self._fatos_criticos = {
            'p1_1', 'p1_2', 'p1_3', 'g1_1', 'g1_2', 'g1_3', 'e1_1', 'g3_1', 't3_1'
        }

    def processar_ciclo_scan(self, telemetria: Dict[str, float]) -> Dict[str, Any]:
        props = self.mapeador.extrair_proposicoes(telemetria)
        fatos_ativos = {k for k, v in props.items() if v}
        trip = any(f in fatos_ativos for f in self._fatos_criticos)
        fatos_inf, trilha = self.motor.forward_chaining(fatos_ativos)
        diagnosticos = sorted(fatos_inf - fatos_ativos)  # só os fatos INFERIDOS, não os brutos
        return {
            "Trip_Ativo": trip,
            "Fatos_Campo": sorted(fatos_ativos),
            "Diagnósticos": diagnosticos,
            "Trilha": trilha
        }


core1 = SCADACoreModulo1()

# ============================================================
# CENÁRIO 1: Sobrepressão + válvula aberta no Tanque de Alta Pressão
# ============================================================
res1 = core1.processar_ciclo_scan({'PT-103': 1050.0, 'XV-103': 1.0})
print("=== CENÁRIO 1: Tanque de Alta Pressão em sobrepressão (PT-103 = 1050 bar, XV-103 aberta) ===")
print("Resultado Scan:", {k: v for k, v in res1.items() if k != "Trilha"})
print(formatar_tabela(res1["Trilha"]))

assert res1["Trip_Ativo"] is True
assert "TRIP_TANQUE_ALTA" in res1["Diagnósticos"]
print("\n[OK] Cenário 1: Trip corretamente ativado e causa-raiz isolada!\n")

# ============================================================
# CENÁRIO 2: Operação normal, abastecimento concluído com sucesso
# ============================================================
res2 = core1.processar_ciclo_scan({'PT-301': 705.0})
print("=== CENÁRIO 2: Abastecimento em andamento, pressão do veículo atinge o setpoint (PT-301 = 705 bar) ===")
print("Resultado Scan:", {k: v for k, v in res2.items() if k != "Trilha"})
print(formatar_tabela(res2["Trilha"]))

assert res2["Trip_Ativo"] is False
assert "ABASTECIMENTO_CONCLUIDO" in res2["Diagnósticos"]
print("\n[OK] Cenário 2: Nenhum trip espúrio, abastecimento concluído normalmente!\n")

print("=== AVALIAÇÃO MÓDULO 1 CONCLUÍDA COM 100% DE SUCESSO! ===")


=== CENÁRIO 1: Tanque de Alta Pressão em sobrepressão (PT-103 = 1050 bar, XV-103 aberta) ===
Resultado Scan: {'Trip_Ativo': True, 'Fatos_Campo': ['p1_3', 'v1_3'], 'Diagnósticos': ['SOBREPRESSAO_TANQUE_ALTA', 'TRIP_TANQUE_ALTA']}
Passo | Regra | Diagnóstico                                                          
------+-------+----------------------------------------------------------------------
1     | R-07  | Pressão do Tanque de Alta Pressão acima do limite (PT-103 > 1000 bar)
2     | R-08  | Corte de Segurança do Tanque de Alta Pressão                         

[OK] Cenário 1: Trip corretamente ativado e causa-raiz isolada!

=== CENÁRIO 2: Abastecimento em andamento, pressão do veículo atinge o setpoint (PT-301 = 705 bar) ===
Resultado Scan: {'Trip_Ativo': False, 'Fatos_Campo': ['p3_1'], 'Diagnósticos': ['ABASTECIMENTO_CONCLUIDO']}
Passo | Regra | Diagnóstico                                                          
------+-------+-------------------------------------------------